<a href="https://colab.research.google.com/github/sankarbaseone/ml-systems-google-prep-/blob/main/day2_jax_tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import jax
import jax.numpy as jnp
import time

@jax.jit
def add(x, y):
    return x + y

# Same shape = cached
x1 = jnp.ones((1024, 1024))
t0 = time.time(); add(x1, x1).block_until_ready()
print("First call:", round(time.time()-t0, 4))

t0 = time.time(); add(x1, x1).block_until_ready()
print("Second call:", round(time.time()-t0, 4))

# Different shape = recompile!
x2 = jnp.ones((2048, 2048))
t0 = time.time(); add(x2, x2).block_until_ready()
print("New shape:", round(time.time()-t0, 4))

# This will FAIL or behave wrong - run it and observe
@jax.jit
def bad_relu(x):
    if x > 0:       # Python if = traced once, WRONG
        return x
    return jnp.zeros_like(x)

# This is correct
@jax.jit
def good_relu(x):
    return jnp.where(x > 0, x, jnp.zeros_like(x))

x = jnp.array([-1.0, 2.0, -3.0, 4.0])
print(good_relu(x))

# Try bad_relu(x) and paste the error
try:
    print(bad_relu(x))
except Exception as e:
    print("Error:", e)

    # Manual loop (slow)
def slow_dot(matrix, vectors):
    return jnp.array([jnp.dot(matrix, v) for v in vectors])

# vmap (fast)
def single_dot(matrix, vector):
    return jnp.dot(matrix, vector)

fast_dot = jax.vmap(single_dot, in_axes=(None, 0))

M = jnp.ones((512, 512))
V = jnp.ones((64, 512))  # 64 vectors

t0 = time.time()
r1 = slow_dot(M, V); r1.block_until_ready()
print("Loop:", round(time.time()-t0, 4))

t0 = time.time()
r2 = fast_dot(M, V); r2.block_until_ready()
print("vmap:", round(time.time()-t0, 4))

First call: 0.0435
Second call: 0.0006
New shape: 0.0409
[0. 2. 0. 4.]
Error: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Loop: 0.6237
vmap: 0.2774


In [1]:
import jax
print(jax.devices())  # Verify TPU before anything else

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
